In [6]:
import os
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
import warnings

# 关闭 Python 常规 warning
warnings.filterwarnings("ignore")

# 关闭 RDKit warning / error log
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')



# =========================================================
# 1. 全局配置
# =========================================================
data_file = "../invertebrates_LC50_unique.xlsx"
smiles_col = "SMILES_Canonical_RDKit"
label_col = "mgperL"

pred_dir = "../k_folds_model/xgb_predictions"   # 你的十折测试集预测值目录
n_splits = 10
GRID_SIZE = 200

# Morgan 指纹参数
radius = 2
n_bits = 2048

# 预测文件名模板：按你的实际文件名修改
pred_file_template = "fold_{fold}_y_pred.npy"

# =========================================================
# 2. 读取数据
# =========================================================
data = pd.read_excel(data_file)
data = data.dropna(subset=[smiles_col, label_col]).copy()

smiles_data = data[smiles_col].tolist()
y_all = np.log1p(data[label_col].values.astype(float))   # 与训练时一致
groups = smiles_data

# =========================================================
# 3. Morgan fingerprint 工具函数
# =========================================================
def smiles_to_morgan_fp(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)

def batch_smiles_to_fps(smiles_list, radius=2, n_bits=2048):
    fps = []
    valid_idx = []
    for i, smi in enumerate(smiles_list):
        fp = smiles_to_morgan_fp(smi, radius=radius, n_bits=n_bits)
        if fp is not None:
            fps.append(fp)
            valid_idx.append(i)
    return fps, np.array(valid_idx, dtype=int)

def max_tanimoto_similarity_to_train(test_fps, train_fps):
    """
    对每个测试样本，计算其与训练集中所有样本的 Tanimoto similarity，
    返回最大 Tanimoto similarity，shape = [n_test]
    """
    max_sims = np.zeros(len(test_fps), dtype=np.float32)
    for i, fp_te in enumerate(test_fps):
        sims = DataStructs.BulkTanimotoSimilarity(fp_te, train_fps)
        max_sims[i] = np.max(sims) if len(sims) > 0 else 0.0
    return max_sims

# =========================================================
# 4. 重建与你训练时一致的 10 折划分
# =========================================================
gkf = GroupKFold(n_splits=n_splits)
fold_splits = list(gkf.split(smiles_data, y_all, groups))

# =========================================================
# 5. 检查预测文件
# =========================================================
for fold_id in range(1, n_splits + 1):
    pred_path = os.path.join(pred_dir, pred_file_template.format(fold=fold_id))
    if not os.path.exists(pred_path):
        raise FileNotFoundError(f"未找到预测文件: {pred_path}")

# =========================================================
# 6. 收集所有折的 Tanimoto AD 分数（max similarity）
# =========================================================
fold_cache = []
all_scores = []

for fold_id, (train_idx, val_idx) in enumerate(fold_splits, start=1):
    train_smiles = [smiles_data[i] for i in train_idx]
    val_smiles   = [smiles_data[i] for i in val_idx]

    y_train = y_all[train_idx]
    y_val   = y_all[val_idx]

    pred_path = os.path.join(pred_dir, pred_file_template.format(fold=fold_id))
    y_pred = np.load(pred_path)

    # 转指纹
    train_fps, train_valid_idx = batch_smiles_to_fps(train_smiles, radius=radius, n_bits=n_bits)
    val_fps, val_valid_idx     = batch_smiles_to_fps(val_smiles, radius=radius, n_bits=n_bits)

    # 若存在无效 SMILES，同步过滤
    if len(train_valid_idx) != len(train_smiles):
        y_train = y_train[train_valid_idx]
        train_smiles = [train_smiles[i] for i in train_valid_idx]

    if len(val_valid_idx) != len(val_smiles):
        y_val = y_val[val_valid_idx]
        y_pred = y_pred[val_valid_idx]
        val_smiles = [val_smiles[i] for i in val_valid_idx]

    # 传统 Tanimoto similarity AD 分数：max similarity
    max_sims = max_tanimoto_similarity_to_train(val_fps, train_fps)

    all_scores.append(max_sims)

    fold_cache.append({
        "fold_id": fold_id,
        "train_smiles": train_smiles,
        "val_smiles": val_smiles,
        "y_train": y_train,
        "y_val": y_val,
        "y_pred": y_pred,
        "ad_score": max_sims
    })

all_scores = np.concatenate(all_scores)

# =========================================================
# 7. 阈值扫描：输出不同阈值下的十折平均 R²_in
#    并增加覆盖率指标：Coverage（保留两位小数）
# =========================================================
score_cuts = np.arange(0.0, 1.0001, 0.05)

results = []

for th in score_cuts:
    fold_r2_list = []
    retained_list = []
    total_list = []

    for cache in fold_cache:
        score = cache["ad_score"]
        y_val = cache["y_val"]
        y_pred = cache["y_pred"]

        in_mask = score >= th

        retained_n = int(in_mask.sum())
        total_n = int(len(in_mask))

        retained_list.append(retained_n)
        total_list.append(total_n)

        if retained_n > 1:
            r2_in = r2_score(y_val[in_mask], y_pred[in_mask])
        else:
            r2_in = np.nan

        fold_r2_list.append(r2_in)

    mean_r2 = np.nanmean(fold_r2_list)
    std_r2 = np.nanstd(fold_r2_list)

    mean_retained = np.mean(retained_list)
    mean_total = np.mean(total_list)
    coverage = mean_retained / mean_total if mean_total > 0 else np.nan

    results.append({
        "threshold": round(float(th), 2),
        "mean_R2_in_10fold": round(float(mean_r2), 4) if not np.isnan(mean_r2) else np.nan,
        "std_R2_in_10fold": round(float(std_r2), 4) if not np.isnan(std_r2) else np.nan,
        "Coverage": round(float(coverage), 2) if not np.isnan(coverage) else np.nan
    })

results_df = pd.DataFrame(results)

print(results_df)
display(results_df)

best_row = results_df.loc[results_df["mean_R2_in_10fold"].idxmax()]

print("=" * 80)
print("最佳阈值结果：")
print(best_row)
print("=" * 80)

    threshold  mean_R2_in_10fold  std_R2_in_10fold  Coverage
0        0.00             0.6357            0.0474      1.00
1        0.05             0.6355            0.0480      0.99
2        0.10             0.6337            0.0479      0.97
3        0.15             0.6349            0.0480      0.97
4        0.20             0.6404            0.0452      0.95
5        0.25             0.6435            0.0494      0.90
6        0.30             0.6430            0.0558      0.84
7        0.35             0.6422            0.0473      0.77
8        0.40             0.6684            0.0486      0.70
9        0.45             0.6600            0.0577      0.62
10       0.50             0.6661            0.0623      0.58
11       0.55             0.6716            0.0733      0.48
12       0.60             0.6839            0.0636      0.42
13       0.65             0.6866            0.0755      0.33
14       0.70             0.6876            0.1140      0.25
15       0.75           

,threshold,mean_R2_in_10fold,std_R2_in_10fold,Coverage
0,0.00,0.6357,0.0474,1.00
1,0.05,0.6355,0.0480,0.99
2,0.10,0.6337,0.0479,0.97
3,0.15,0.6349,0.0480,0.97
4,0.20,0.6404,0.0452,0.95
5,0.25,0.6435,0.0494,0.90
6,0.30,0.6430,0.0558,0.84
7,0.35,0.6422,0.0473,0.77
8,0.40,0.6684,0.0486,0.70
9,0.45,0.6600,0.0577,0.62


最佳阈值结果：
threshold            0.7500
mean_R2_in_10fold    0.6905
std_R2_in_10fold     0.1286
Coverage             0.1900
Name: 15, dtype: float64


In [7]:
results_df.to_csv("./Tanimoto similarity.csv")